# R01 · Will it meet its deadlines? Rate-monotonic and response-time analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/05_realtime_linux/R01_will_it_meet_its_deadlines_rate_monotonic_and_re.ipynb)

**Track:** Real-time and Linux · **Skill axis (sheet):** Real-time and Linux · **Time:** about 50 min

**Skill Hub concepts:** Rate-monotonic scheduling; Response-time analysis; Per-core utilization budget

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("R01")

## Map: periodic tasks on a fixed-priority preemptive scheduler (FreeRTOS, Zephyr, SCHED_FIFO)

Task $i$: worst-case execution time $C_i$, period $T_i$, deadline $D_i = T_i$. Utilization $U = \sum C_i/T_i$.

| Test | Rule | Verdict |
|---|---|---|
| Rate-monotonic priorities | shorter period = higher priority | optimal among fixed-priority schemes for $D = T$ |
| Liu and Layland bound | $U \le n(2^{1/n} - 1)$ (0.828 for 2, 0.780 for 3, 0.693 as $n \to \infty$) | **sufficient**: below = schedulable, above = unknown |
| Necessary | $U \le 1$ | above 1 = impossible |
| Response-time analysis (exact) | $R_i = C_i + \sum_{j \in hp(i)} \left\lceil R_i / T_j \right\rceil C_j$, iterate from $R_i = C_i$ | schedulable iff every $R_i \le D_i$ |

Multicore (partitioned): assign each task to one core (first-fit decreasing by utilization), then run the
single-core test per core. Keep a margin (for example 70% per core) for interrupts and growth.

In [ ]:
import math

## Task 1: utilization tests
Tasks are `(C, T)` tuples.

In [ ]:
def ll_bound(n):
    raise NotImplementedError  # TODO

def rm_test(tasks):
    """'schedulable' if U <= bound, 'unschedulable' if U > 1, else 'unknown'."""
    raise NotImplementedError  # TODO

lab.check("utilization", ll_bound, rm_test)

In [ ]:
%%writefile r01_utilization.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/r01.h
// Grade it with lab.check("utilization", "r01_utilization.c"). For C++, name the file .cpp in both places.
#include "r01.h"

double ll_bound(int n) { (void)n; return 0; /* TODO: Liu and Layland */ }

int rm_test(const double *c, const double *t, size_t n) {
    (void)c; (void)t; (void)n;
    return RM_UNKNOWN; /* TODO */
}

In [ ]:
# lab.check("utilization", "r01_utilization.c")  # uncomment after writing the C/C++ cell above

## Task 2: exact response times
Sort by period (rate-monotonic). Return the list of response times in that order, with `None` for any task that
misses its deadline.

In [ ]:
def response_times(tasks):
    raise NotImplementedError  # TODO

lab.check("rta", response_times)

In [ ]:
%%writefile r01_rta.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/r01.h
// Grade it with lab.check("rta", "r01_rta.c"). For C++, name the file .cpp in both places.
#include "r01.h"

void response_times(const uint32_t *c, const uint32_t *t, size_t n, int32_t *r_out) {
    (void)c; (void)t;
    for (size_t i = 0; i < n; ++i) r_out[i] = -1; /* TODO: sort by period, iterate R = C + sum ceil(R/Tj) Cj */
}

In [ ]:
# lab.check("rta", "r01_rta.c")  # uncomment after writing the C/C++ cell above

## Task 3: partition onto cores
First-fit decreasing by utilization: take tasks from largest $C/T$ to smallest, put each on the first core where the
response-time test still passes for every task on that core. Return a list of task lists (one per core), or `None`.

In [ ]:
def partition_ffd(tasks, n_cores):
    raise NotImplementedError  # TODO

lab.check("partition", partition_ffd)

In [ ]:
%%writefile r01_partition.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/r01.h
// Grade it with lab.check("partition", "r01_partition.c", "r01_rta.c"). For C++, name the file .cpp in both places.
#include "r01.h"

bool partition_ffd(const uint32_t *c, const uint32_t *t, size_t n, int n_cores, int *core_of) {
    (void)c; (void)t; (void)n_cores;
    for (size_t i = 0; i < n; ++i) core_of[i] = 0;
    return false; /* TODO: sort by utilization, first core where response_times() still meets every deadline */
}

In [ ]:
# lab.check("partition", "r01_partition.c", "r01_rta.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** Liu and Layland bound for 3 tasks (4 decimals)?

**M2 (Analyze).** Tasks (1, 4), (2, 6), (3, 12) in milliseconds. Worst-case response time of the lowest-priority task?

**M3 (Apply).** Utilizations 0.5, 0.4, 0.3, 0.3 placed first-fit decreasing on 2 cores with a limit of 1.0 per core.
Load of the busiest core?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** A task set has U = 0.83 with 3 tasks, above the 0.78 bound. Conclusion?
- a) it will miss deadlines
- b) unknown from the bound alone; run response-time analysis, which is exact
- c) switch to round-robin

**K2 (Evaluate).** On a Cortex-M: a 1 kHz current loop (0.25 ms), 100 Hz telemetry (2 ms), CAN handling at 500 Hz
(0.3 ms). Assign priorities, check schedulability, and say what must be added to the analysis for a real system.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `r01.h`).
3. In `roboprep/checks/r01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()